# Predict — Faster R-CNN on SAR Images

## 0. Setup

In [ ]:
import sys, os, random
import json, torch
import yaml
from PIL import Image
from IPython.display import display
from pathlib import Path

ROOT = Path(os.getcwd()).parent 
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from scripts.predict import run_on_image
from models.model import get_model
from data.dataset import SAR_ATR_Dataset
from data.transforms import CocoToFasterRCNN


## 1. Load Weights and Class Names

Make sure `faster_rcnn.pt` and `classes.json` are placed at the correct paths as described in the README.  
Pre-trained weights, and `classes.json` are available at: https://drive.google.com/drive/folders/1aWoHLGQUyak-8OvW0_y0BhOG2OJEFAW1

In [ ]:
WEIGHTS_PATH = ROOT / "experiments" / "models" / "faster_rcnn.pt"
CLASSES_PATH = ROOT / "experiments" / "config" / "classes.json"

assert WEIGHTS_PATH.exists(), f"Weights not found: {WEIGHTS_PATH}"
assert CLASSES_PATH.exists(), f"classes.json not found: {CLASSES_PATH}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
num_classes = checkpoint["roi_heads.box_predictor.cls_score.weight"].shape[0]
model = get_model(num_classes)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
print(f"Model loaded — {num_classes} classes (including background)")

with open(CLASSES_PATH) as f:
    class_names = json.load(f)
print(f"Classes: {list(class_names.values())}")

## 2. Test Dataset (optional)

Required only for Section 3. Skip if you only want to run inference on a custom image.  
Make sure the dataset is placed at the correct paths as described in the README.

In [ ]:
dataset = None

with open(ROOT / "experiments" / "config" / "config.yaml") as f:
    config = yaml.safe_load(f)

img_dir  = ROOT / config["data"]["images"]["test"]["img_dir"]
ann_file = ROOT / config["data"]["annotations"]["test"]["ann_file"]

if img_dir.exists() and ann_file.exists():
    dataset = SAR_ATR_Dataset(root=str(img_dir), annFile=str(ann_file), transforms=CocoToFasterRCNN())
    print(f"Dataset loaded — {len(dataset)} test images")
else:
    print("Test dataset not found — Section 3 will be skipped.")

## 3. Inference on Test Dataset

Re-run the cell to sample a different image.  
Green: ground truth — Red: predictions

In [ ]:
THRESHOLD = 0.5

if dataset is None:
    print("Dataset not loaded — skipping.")
else:
    idx = random.randint(0, len(dataset) - 1)
    _, target = dataset[idx]
    image_info = dataset.coco.loadImgs(dataset.ids[idx])[0]
    image_path = os.path.join(dataset.root, image_info["file_name"])
    ground_truth = {"boxes": target["boxes"], "labels": target["labels"]}

    save_path = run_on_image(image_path, model, device, threshold=THRESHOLD,
                             ground_truth=ground_truth, class_names=class_names)
    display(Image.open(save_path).resize((256, 256), Image.NEAREST))

## 4. Inference on a Custom Image

In [ ]:
IMAGE_PATH = "/path/to/your/image.png"  # update this
THRESHOLD = 0.5

save_path = run_on_image(IMAGE_PATH, model, device, threshold=THRESHOLD,
                         class_names=class_names)
display(Image.open(save_path).resize((256, 256), Image.NEAREST))